In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import os
import sys
import logging
import zipfile

import pandas as pd
import numpy as np
import datasets

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split

# ========== Kaggle路径常量 ==========
INPUT_DIR = "/kaggle/input/competitions/word2vec-nlp-tutorial"
WORKING_DIR = "/kaggle/working"

TRAIN_ZIP = os.path.join(INPUT_DIR, "labeledTrainData.tsv.zip")
TEST_ZIP = os.path.join(INPUT_DIR, "testData.tsv.zip")

os.makedirs(os.path.join(WORKING_DIR, "checkpoint"), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, "result"), exist_ok=True)


def read_tsv_from_zip(zip_path, tsv_filename):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(tsv_filename) as f:
            df = pd.read_csv(f, header=0, delimiter="\t", quoting=3)
    return df


if __name__ == '__main__':
    logging.basicConfig(
        format='%(asctime)s - %(message)s',
        datefmt='%H:%M:%S',
        level=logging.INFO,
        stream=sys.stdout
    )
    logging.getLogger("transformers").setLevel(logging.WARNING)
    logging.getLogger("datasets").setLevel(logging.WARNING)

    print("==== 1.加载数据集 ====")
    train = read_tsv_from_zip(TRAIN_ZIP, "labeledTrainData.tsv")
    test = read_tsv_from_zip(TEST_ZIP, "testData.tsv")
    print(f"原始训练集:{len(train)}  测试集:{len(test)}")

    train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)
    print(f"划分训练集:{len(train_df)} 验证集:{len(val_df)}")

    train_dataset = datasets.Dataset.from_dict({"label": train_df["sentiment"], "text": train_df["review"]})
    val_dataset = datasets.Dataset.from_dict({"label": val_df["sentiment"], "text": val_df["review"]})
    test_dataset = datasets.Dataset.from_dict({"text": test["review"]})

    print("==== 2.文本token化 ====")
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    def preprocess_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512)

    tokenized_train = train_dataset.map(preprocess_fn, batched=True)
    tokenized_val = val_dataset.map(preprocess_fn, batched=True)
    tokenized_test = test_dataset.map(preprocess_fn, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = np.sum(preds == labels) / len(labels)
        return {"accuracy": float(acc)}

    print("==== 3.开始训练 ====")
    training_args = TrainingArguments(
        output_dir=os.path.join(WORKING_DIR, "checkpoint"),
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=os.path.join(WORKING_DIR, "logs"),
        logging_steps=10000,
        save_strategy="no",
        eval_strategy="epoch",
        report_to="none",
        fp16=True,
        disable_tqdm=False,
    )

    # 修复：删除 tokenizer=tokenizer 参数
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    print("==== 4.训练完成，开始预测测试集 ====")

    pred_out = trainer.predict(tokenized_test)
    test_pred = np.argmax(pred_out.predictions, axis=-1).flatten()

    result_df = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    out_file = os.path.join(WORKING_DIR, "result/bert_trainer.csv")
    result_df.to_csv(out_file, index=False, quoting=3)
    print(f"==== 全部完成，结果保存至: {out_file} ====")


==== 1.加载数据集 ====
原始训练集:25000  测试集:25000
划分训练集:20000 验证集:5000
==== 2.文本token化 ====
03:21:23 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
03:21:23 - HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


03:21:23 - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

03:21:23 - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
03:21:23 - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
03:21:23 - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
03:21:23 - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
03:21:23 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
03:21:23 - HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"


vocab.txt: 0.00B [00:00, ?B/s]

...

==== 3.开始训练 ====


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.406537,0.922200
2,No log,0.482370,0.924000
3,No log,0.558623,0.933600


==== 4.训练完成，开始预测测试集 ====


==== 全部完成，结果保存至: /kaggle/working/result/bert_trainer.csv ====


In [3]:
import os
import sys
import zipfile
import re
import pandas as pd
import numpy as np
import datasets
import torch
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split

# ========== Kaggle路径常量 ==========
INPUT_DIR = "/kaggle/input/competitions/word2vec-nlp-tutorial"
WORKING_DIR = "/kaggle/working"

TRAIN_ZIP = os.path.join(INPUT_DIR, "labeledTrainData.tsv.zip")
TEST_ZIP = os.path.join(INPUT_DIR, "testData.tsv.zip")

os.makedirs(os.path.join(WORKING_DIR, "checkpoint"), exist_ok=True)
os.makedirs(os.path.join(WORKING_DIR, "result"), exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 简单文本清洗：去除html标记，IMDB原始数据有<br />标签，严重干扰BERT
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def read_tsv_from_zip(zip_path, tsv_filename):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(tsv_filename) as f:
            df = pd.read_csv(f, header=0, delimiter="\t")
    return df

if __name__ == '__main__':
    print("==== 1.加载数据集 + 文本清洗 ====")
    train = read_tsv_from_zip(TRAIN_ZIP, "labeledTrainData.tsv")
    test = read_tsv_from_zip(TEST_ZIP, "testData.tsv")

    train["review"] = train["review"].apply(clean_text)
    test["review"] = test["review"].apply(clean_text)

    print(f"原始训练集:{len(train)}  测试集:{len(test)}")

    # stratify分层划分，保证验证集正负样本均衡
    train_df, val_df = train_test_split(train, test_size=0.2, random_state=SEED, stratify=train["sentiment"])
    print(f"划分训练集:{len(train_df)} 验证集:{len(val_df)}")

    train_dataset = datasets.Dataset.from_dict({"label": train_df["sentiment"], "text": train_df["review"]})
    val_dataset = datasets.Dataset.from_dict({"label": val_df["sentiment"], "text": val_df["review"]})
    test_dataset = datasets.Dataset.from_dict({"text": test["review"]})

    print("==== 2.文本token化 ====")
    # 可选更强权重：bert-base-uncased-whole-word-masking
    model_name = "bert-base-uncased"
    tokenizer = BertTokenizerFast.from_pretrained(model_name)

    def preprocess_fn(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=384,    # 关键优化，不用512，降低padding噪声
            padding="max_length"
        )

    tokenized_train = train_dataset.map(preprocess_fn, batched=True)
    tokenized_val = val_dataset.map(preprocess_fn, batched=True)
    tokenized_test = test_dataset.map(preprocess_fn, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = np.sum(preds == labels) / len(labels)
        return {"accuracy": float(acc)}

    print("==== 3.开始训练 ====")
    training_args = TrainingArguments(
        output_dir=os.path.join(WORKING_DIR, "checkpoint"),
        num_train_epochs=2,                # 降为2轮，防止过拟合IMDB
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,                # BERT finetune关键调参，2e‑5最优
        warmup_ratio=0.1,                  # 使用比例代替固定步数，更稳定
        weight_decay=0.01,
        logging_steps=500,
        save_strategy="no",
        eval_strategy="epoch",
        report_to="none",
        fp16=True,                         # T4 GPU打开，加速+省显存
        disable_tqdm=False,
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    val_result = trainer.evaluate()
    print(f"==== 验证集最终准确率: {val_result['eval_accuracy']:.4f} ====")

    print("==== 4.训练完成，预测测试集（输出概率便于后续集成） ====")
    pred_out = trainer.predict(tokenized_test)
    logits = pred_out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    test_pred = np.argmax(probs, axis=-1)

    result_df = pd.DataFrame({
        "id": test["id"],
        "sentiment": test_pred,
        "prob_pos": probs[:,1]
    })
    out_file = os.path.join(WORKING_DIR, "result/bert_trainer.csv")
    submit_df = result_df[["id","sentiment"]]
    submit_df.to_csv(out_file, index=False)
    print(f"==== 提交文件保存至: {out_file} ====")
    print(f"预测样本数量: {len(submit_df)}")
    print(f"正样本预测占比: {(test_pred.sum() / len(test_pred)):.3f}")


==== 1.加载数据集 + 文本清洗 ====
原始训练集:25000  测试集:25000
划分训练集:20000 验证集:5000
==== 2.文本token化 ====
04:44:13 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
04:44:13 - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
04:44:13 - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
04:44:13 - HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
04:44:13 - HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

04:44:37 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
04:44:38 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
04:44:38 - HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/model.safetensors "HTTP/1.1 302 Found"


...

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

==== 3.开始训练 ====


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,1.263557,0.410545,0.921000
2,0.647458,0.429236,0.928400


==== 验证集最终准确率: 0.9284 ====
==== 4.训练完成，预测测试集（输出概率便于后续集成） ====
==== 提交文件保存至: /kaggle/working/result/bert_trainer.csv ====
预测样本数量: 25000
正样本预测占比: 0.507


In [ ]:
|